# 03: CSR Interaction Matrix & Item-Item Collaborative Filtering
This notebook demonstrates constructing an implicit interaction matrix with event weights and damped logarithmic scaling, computing sparse cosine item similarity, and generating explainable recommendations.

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

# Robust root resolver whether run from notebooks/ or workspace root
_curr = Path('.').resolve()
PROJECT_ROOT = _curr if (_curr / 'src').exists() else (_curr.parent if (_curr.parent / 'src').exists() else (_curr.parents[1] if len(_curr.parents) > 1 and (_curr.parents[1] / 'src').exists() else _curr))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Switch working directory to project root so all relative paths ('data/...', 'artifacts/...') resolve seamlessly
os.chdir(PROJECT_ROOT)

from src.features.interaction_features import InteractionMatrixBuilder
from src.recommenders.item_item import ItemItemRecommender


## 1. Build Damped Interaction Matrix

In [2]:
train_df = pd.read_parquet('data/processed/train_events.parquet')
id_builder = InteractionMatrixBuilder().fit_mappings(train_df)
csr_mat, agg_df = id_builder.build_matrix(train_df)
print(f"Matrix shape: {csr_mat.shape}, Non-zeros: {csr_mat.nnz:,}")


2026-10-10 10:49:28 [INFO] [interaction_features:50] Built ID mappings: 2996 unique users, 800 unique items.
2026-10-10 10:49:28 [INFO] [interaction_features:55] Computing weighted implicit interactions and building CSR matrix...
2026-10-10 10:49:28 [INFO] [interaction_features:87] Interaction matrix assembled: (2996, 800) with 30,869 non-zero entries.
Matrix shape: (2996, 800), Non-zeros: 30,869


## 2. Train Item-Item Collaborative Filtering

In [3]:
item_model = ItemItemRecommender(k_nearest=50).fit(csr_mat, id_builder)
test_prod_id = list(id_builder.item_to_idx.keys())[0]
similar_items = item_model.get_similar_items(test_prod_id, n=5)
print(f"Top-5 items similar to Product #{test_prod_id}:")
for pid, sim in similar_items:
    print(f"  - Product #{pid}: Similarity = {sim:.4f}")


2026-10-10 10:49:32 [INFO] [item_item_recommender:40] Training Item-Item CF with top-50 neighbors...
2026-10-10 10:49:32 [INFO] [item_item_recommender:91] Item-Item similarity graph computed across 800 items.
Top-5 items similar to Product #1000:
  - Product #1001: Similarity = 0.4137
  - Product #1002: Similarity = 0.3345
  - Product #1003: Similarity = 0.2740
  - Product #1004: Similarity = 0.2528
  - Product #1006: Similarity = 0.2298


## 3. Recommend with Explainability Traces

In [4]:
test_cust_id = list(id_builder.user_to_idx.keys())[0]
recs = item_model.recommend(test_cust_id, n=5)
print(f"Recommendations for Customer #{test_cust_id}:")
for pid, score in recs:
    explanation = item_model.explain(test_cust_id, pid)
    print(f"  - Product #{pid} (Score: {score:.3f}): {explanation['explanation']}")


Recommendations for Customer #20000:
  - Product #1001 (Score: 1.000): Recommended because it is highly similar (0.41) to product 1000 previously purchased/viewed.
  - Product #1565 (Score: 0.809): Recommended because it is highly similar (0.35) to product 1609 previously purchased/viewed.
  - Product #1003 (Score: 0.579): Recommended because it is highly similar (0.27) to product 1000 previously purchased/viewed.
  - Product #1029 (Score: 0.538): Recommended because it is highly similar (0.16) to product 1609 previously purchased/viewed.
  - Product #1602 (Score: 0.460): Recommended because it is highly similar (0.18) to product 1609 previously purchased/viewed.
